# Electrostatics validation suite

Closed-form **validation-class** electrostatic identities, computed with the pure-Python helpers in `radia_mcp.radia_ngsolve.electrostatics` / `.force`. Each section reconciles several equivalent textbook expressions for the same capacitance / energy / Maxwell-stress force and prints the agreement (the printed `[checks]` are executed assertions). No mesh, no solver -- these are the analytic reference layer the FEM/BEM electrostatic post-processing is checked against.

*Corpus kept at `examples/electrostatics/` (cited by `radia_mcp.radia_ngsolve.force` knowledge); this notebook is the rendered showcase.*

## 1. Parallel-plate force identity

`C = eps A/d`, `W = 1/2 C V^2`, pressure `p = 1/2 eps (V/d)^2`, force `F = pA = W/d` -- four textbook views of the same normal electrostatic force, all reconciled.

In [1]:
import sys, os
sys.argv = ["notebook"]

"""Validation-class parallel-plate electrostatic force identity.

This example connects three equivalent public textbook views of the same
normal electrostatic force:

    C = eps A / d
    W = 0.5 C V^2
    p = 0.5 eps (V / d)^2
    F = p A = W / d

Run:

    python examples/electrostatics/validation_parallel_plate_electrostatic_force.py
"""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path


HERE = Path(os.path.join(os.getcwd(), 'notebook.py')).resolve().parent
REPO = HERE.parents[1]
SRC = REPO / "packages" / "radia-mcp" / "src"
# radia_mcp is pip-installed (editable); no sys.path shim needed

from radia_mcp.radia_ngsolve.electrostatics import (  # noqa: E402
    EPS0,
    parallel_plate_capacitor_energy_force,
)
from radia_mcp.radia_ngsolve.force import electrostatic_traction_summary  # noqa: E402


OUT_JSON = HERE / "validation_parallel_plate_electrostatic_force_summary.json"

EPS_R = 2.5
AREA_M2 = 2.0e-4
GAP_M = 0.5e-3
VOLTAGE_V = 120.0


def _assert_close(value: float, expected: float, rel: float = 1.0e-12) -> float:
    err = abs(value - expected)
    if err > rel * max(abs(value), abs(expected), 1.0):
        raise AssertionError(f"{value!r} != {expected!r}")
    return err


def build_summary() -> dict:
    out = parallel_plate_capacitor_energy_force(EPS_R, AREA_M2, GAP_M, VOLTAGE_V)
    field = VOLTAGE_V / GAP_M
    eps = EPS0 * EPS_R
    pressure = 0.5 * eps * field * field
    force = pressure * AREA_M2
    traction = electrostatic_traction_summary(
        (0.0, 0.0, field),
        (0.0, 0.0, 1.0),
        area_m2=AREA_M2,
        eps=eps,
    )
    half_gap = parallel_plate_capacitor_energy_force(EPS_R, AREA_M2, 0.5 * GAP_M, VOLTAGE_V)

    checks = {
        "capacitance_error_F": _assert_close(out["C"], eps * AREA_M2 / GAP_M),
        "energy_error_J": _assert_close(out["energy"], 0.5 * out["C"] * VOLTAGE_V * VOLTAGE_V),
        "pressure_error_Pa": _assert_close(out["pressure_Pa"], pressure),
        "force_error_N": _assert_close(out["force"], force),
        "force_energy_gap_error_N": _assert_close(out["force"], out["energy"] / GAP_M),
        "traction_pressure_error_Pa": _assert_close(
            traction["normal_traction_Pa"],
            out["pressure_Pa"],
        ),
        "traction_force_error_N": _assert_close(traction["force_N"][2], out["force"]),
        "half_gap_force_ratio_error": _assert_close(half_gap["force"] / out["force"], 4.0),
    }

    return {
        "kind": "parallel_plate_electrostatic_force",
        "validation_class": True,
        "inputs": {
            "eps_r": EPS_R,
            "area_m2": AREA_M2,
            "gap_m": GAP_M,
            "voltage_V": VOLTAGE_V,
        },
        "parallel_plate": out,
        "maxwell_traction": traction,
        "half_gap_force_N": half_gap["force"],
        "checks": checks,
    }


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--out", type=Path, default=OUT_JSON)
    args = parser.parse_args()

    summary = build_summary()
    args.out.parent.mkdir(parents=True, exist_ok=True)
    args.out.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")

    plate = summary["parallel_plate"]
    print("[Parallel-plate electrostatic force]")
    print(f"  capacitance: {plate['C']:.12g} F")
    print(f"  field:       {plate['electric_field_V_per_m']:.12g} V/m")
    print(f"  pressure:    {plate['pressure_Pa']:.12g} Pa")
    print(f"  force:       {plate['force']:.12g} N")
    print("[checks]")
    for key, value in summary["checks"].items():
        print(f"  {key}: {value:.3e}")
    return 0


if True:
    main()


[Parallel-plate electrostatic force]
  capacitance: 8.8541878128e-12 F
  field:       240000 V/m
  pressure:    0.637501522522 Pa
  force:       0.000127500304504 N
[checks]
  capacitance_error_F: 0.000e+00
  energy_error_J: 0.000e+00
  pressure_error_Pa: 0.000e+00
  force_error_N: 0.000e+00
  force_energy_gap_error_N: 2.711e-20
  traction_pressure_error_Pa: 1.110e-16
  traction_force_error_N: 2.711e-20
  half_gap_force_ratio_error: 0.000e+00


## 2. Coaxial capacitor force

The cylindrical analogue: capacitance, stored energy, Maxwell pressure and the capacitance-gradient force all collapse to the same closed form.

In [2]:
import sys, os
sys.argv = ["notebook"]

"""Validation-class coaxial capacitor electrostatic force.

Run:

    python examples/electrostatics/validation_coaxial_capacitor_force.py

The coaxial capacitor is the cylindrical analogue of the parallel-plate force
gate: capacitance, stored energy, Maxwell pressure, and capacitance-gradient
force all collapse to the same closed form.
"""

from __future__ import annotations

import json
import math
import sys
from pathlib import Path

ROOT = Path(os.path.join(os.getcwd(), 'notebook.py')).resolve().parents[2]
SRC = ROOT / "packages" / "radia-mcp" / "src"
# radia_mcp is pip-installed (editable); no sys.path shim needed

from radia_mcp.radia_ngsolve.electrostatics import (  # noqa: E402
    EPS0,
    capacitance_gradient_force_summary,
    coaxial_capacitor_energy_force,
)

OUT_JSON = Path(os.path.join(os.getcwd(), 'notebook.py')).with_name("validation_coaxial_capacitor_force_summary.json")

EPS_R = 2.5
R_INNER = 0.01
R_OUTER = 0.03
LENGTH = 0.2
VOLTAGE = 120.0


def _assert_close(actual: float, expected: float, name: str, rel: float = 1.0e-12) -> float:
    scale = max(1.0, abs(expected))
    err = abs(actual - expected)
    if err > rel * scale:
        raise AssertionError(f"{name}: got {actual:.16e}, expected {expected:.16e}")
    return err


def main() -> None:
    out = coaxial_capacitor_energy_force(EPS_R, R_INNER, R_OUTER, LENGTH, VOLTAGE)
    eps = EPS0 * EPS_R
    log_ratio = math.log(R_OUTER / R_INNER)
    expected_c = 2.0 * math.pi * eps * LENGTH / log_ratio
    expected_inner_force = (
        math.pi * eps * LENGTH * VOLTAGE * VOLTAGE / (R_INNER * log_ratio * log_ratio)
    )
    expected_outer_force = (
        -math.pi * eps * LENGTH * VOLTAGE * VOLTAGE / (R_OUTER * log_ratio * log_ratio)
    )

    inner_grad = capacitance_gradient_force_summary(
        out["C"],
        out["dCdr_inner_F_per_m"],
        voltage_V=VOLTAGE,
    )
    outer_grad = capacitance_gradient_force_summary(
        out["C"],
        out["dCdr_outer_F_per_m"],
        voltage_V=VOLTAGE,
    )

    checks = {
        "capacitance_abs_error_F": _assert_close(out["C"], expected_c, "capacitance"),
        "inner_force_abs_error_N": _assert_close(
            out["inner_radius_force_N"],
            expected_inner_force,
            "inner force",
        ),
        "outer_force_abs_error_N": _assert_close(
            out["outer_radius_force_N"],
            expected_outer_force,
            "outer force",
        ),
        "inner_pressure_force_error_N": _assert_close(
            out["inner_radius_force_N"],
            out["inner_pressure_area_force_N"],
            "inner pressure force",
        ),
        "outer_pressure_force_error_N": _assert_close(
            out["outer_radius_force_N"],
            out["outer_pressure_area_force_N"],
            "outer pressure force",
        ),
        "inner_gradient_force_error_N": _assert_close(
            out["inner_radius_force_N"],
            inner_grad["fixed_voltage_force_N"],
            "inner gradient force",
        ),
        "outer_gradient_force_error_N": _assert_close(
            out["outer_radius_force_N"],
            outer_grad["fixed_voltage_force_N"],
            "outer gradient force",
        ),
    }

    summary = {
        "kind": "coaxial_capacitor_force_validation",
        "parameters": {
            "eps_r": EPS_R,
            "r_inner_m": R_INNER,
            "r_outer_m": R_OUTER,
            "length_m": LENGTH,
            "voltage_V": VOLTAGE,
        },
        "closed_form": out,
        "inner_radius_capacitance_gradient": inner_grad,
        "outer_radius_capacitance_gradient": outer_grad,
        "checks": checks,
    }
    OUT_JSON.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    print("[Coaxial capacitor electrostatic force]")
    print(f"  C_F: {out['C']:.12e}")
    print(f"  inner_radius_force_N: {out['inner_radius_force_N']:.12e}")
    print(f"  outer_radius_force_N: {out['outer_radius_force_N']:.12e}")
    print(f"  pressure_inner_Pa: {out['pressure_inner_Pa']:.12e}")
    print(f"  wrote {OUT_JSON}")


if True:
    main()


[Coaxial capacitor electrostatic force]
  C_F: 2.531944314943e-11
  inner_radius_force_N: 1.659366025269e-05
  outer_radius_force_N: -5.531220084230e-06
  pressure_inner_Pa: 1.320481526602e-03
  wrote S:\Radia\01_GitHub\docs\electrostatics\validation_coaxial_capacitor_force_summary.json


## 3. Force from a capacitance gradient

Fixed voltage `F_x = 1/2 V^2 dC/dx`; fixed charge `F_x = 1/2 Q^2 C^-2 dC/dx`. Sign is along increasing x (attraction toward the smaller gap).

In [3]:
import sys, os
sys.argv = ["notebook"]

"""Validation-class electrostatic force from a capacitance gradient.

For a displacement coordinate x and capacitance C(x),

    fixed voltage: F_x = 0.5 V^2 dC/dx
    fixed charge:  F_x = 0.5 Q^2 C^-2 dC/dx

The sign is along increasing x.  If x is a gap or height, dC/dx is usually
negative, so the force is negative: attraction toward the smaller gap.

Run:

    python examples/electrostatics/validation_capacitance_gradient_force.py
"""

from __future__ import annotations

import argparse
import json
import math
import sys
from pathlib import Path


HERE = Path(os.path.join(os.getcwd(), 'notebook.py')).resolve().parent
REPO = HERE.parents[1]
SRC = REPO / "packages" / "radia-mcp" / "src"
# radia_mcp is pip-installed (editable); no sys.path shim needed

from radia_mcp.radia_ngsolve.electrostatics import (  # noqa: E402
    EPS0,
    capacitance_gradient_force_summary,
    parallel_plate_capacitor_energy_force,
    sphere_above_plane_capacitance,
)


OUT_JSON = HERE / "validation_capacitance_gradient_force_summary.json"

EPS_R = 2.5
AREA_M2 = 2.0e-4
GAP_M = 5.0e-4
VOLTAGE_V = 120.0
SPHERE_RADIUS_M = 0.01
SPHERE_HEIGHT_M = 2.0 * SPHERE_RADIUS_M


def _assert_close(actual: float, expected: float, *, rtol: float = 1.0e-12, atol: float = 1.0e-18) -> float:
    error = abs(actual - expected)
    if error > max(atol, rtol * max(abs(actual), abs(expected))):
        raise AssertionError(f"{actual!r} != {expected!r}")
    return error


def _sphere_capacitance(height_m: float) -> float:
    return sphere_above_plane_capacitance(
        SPHERE_RADIUS_M,
        height_m,
        eps_r=1.0,
        n_terms=250,
    )["C"]


def build_summary() -> dict:
    plate = parallel_plate_capacitor_energy_force(EPS_R, AREA_M2, GAP_M, VOLTAGE_V)
    dcdgap = -plate["C"] / GAP_M
    charge = plate["C"] * VOLTAGE_V
    gap_force = capacitance_gradient_force_summary(
        plate["C"],
        dcdgap,
        voltage_V=VOLTAGE_V,
        charge_C=charge,
    )
    closing_force = capacitance_gradient_force_summary(
        plate["C"],
        -dcdgap,
        voltage_V=VOLTAGE_V,
        charge_C=charge,
    )
    plate_checks = {
        "gap_force_matches_minus_pressure_force_N": _assert_close(
            gap_force["fixed_voltage_force_N"],
            -plate["force"],
        ),
        "closing_force_matches_pressure_force_N": _assert_close(
            closing_force["fixed_voltage_force_N"],
            plate["force"],
        ),
        "fixed_voltage_charge_route_abs_error_N": _assert_close(
            gap_force["fixed_voltage_force_N"],
            gap_force["fixed_charge_force_N"],
        ),
    }

    step = 1.0e-5 * SPHERE_HEIGHT_M
    c0 = _sphere_capacitance(SPHERE_HEIGHT_M)
    c_minus = _sphere_capacitance(SPHERE_HEIGHT_M - step)
    c_plus = _sphere_capacitance(SPHERE_HEIGHT_M + step)
    dcdh = (c_plus - c_minus) / (2.0 * step)
    sphere_force = capacitance_gradient_force_summary(
        c0,
        dcdh,
        voltage_V=VOLTAGE_V,
    )
    if not sphere_force["fixed_voltage_force_N"] < 0.0:
        raise AssertionError("sphere height force should be negative: attraction toward the plane")

    return {
        "kind": "capacitance_gradient_force_validation",
        "validation_class": True,
        "force_learning": "fixed-voltage electrostatic force is +0.5 V^2 dC/dx along the chosen coordinate",
        "parallel_plate": {
            "eps_r": EPS_R,
            "area_m2": AREA_M2,
            "gap_m": GAP_M,
            "voltage_V": VOLTAGE_V,
            "closed_form": plate,
            "dCdgap_F_per_m": dcdgap,
            "gap_coordinate_force": gap_force,
            "closing_coordinate_force": closing_force,
            "checks": plate_checks,
        },
        "sphere_ground_plane": {
            "radius_m": SPHERE_RADIUS_M,
            "height_m": SPHERE_HEIGHT_M,
            "voltage_V": VOLTAGE_V,
            "step_m": step,
            "capacitance_F": c0,
            "dCdh_F_per_m": dcdh,
            "height_coordinate_force": sphere_force,
            "attractive_force_magnitude_N": -sphere_force["fixed_voltage_force_N"],
        },
    }


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--out", type=Path, default=OUT_JSON)
    args = parser.parse_args()

    summary = build_summary()
    args.out.parent.mkdir(parents=True, exist_ok=True)
    args.out.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")

    plate = summary["parallel_plate"]
    sphere = summary["sphere_ground_plane"]
    print("[Capacitance-gradient electrostatic force]")
    print(f"  plate_gap_force_N: {plate['gap_coordinate_force']['fixed_voltage_force_N']:.12g}")
    print(f"  plate_closing_force_N: {plate['closing_coordinate_force']['fixed_voltage_force_N']:.12g}")
    print(f"  plate_pressure_force_N: {plate['closed_form']['force']:.12g}")
    print(f"  sphere_height_force_N: {sphere['height_coordinate_force']['fixed_voltage_force_N']:.12g}")
    print(f"  sphere_attraction_magnitude_N: {sphere['attractive_force_magnitude_N']:.12g}")
    return 0


if True:
    main()


[Capacitance-gradient electrostatic force]
  plate_gap_force_N: -0.000127500304504
  plate_closing_force_N: 0.000127500304504
  plate_pressure_force_N: 0.000127500304504
  sphere_height_force_N: -1.93339093404e-07
  sphere_attraction_magnitude_N: 1.93339093404e-07


## 4. Sphere over a grounded plane

Image-series capacitance on a height sweep: slow convergence near the plane, `C -> 4 pi eps0 a (1 + a/2h + ...)` far away, attraction from `-dC/dh > 0`.

In [4]:
import sys, os
sys.argv = ["notebook"]

"""Validation-class sphere-over-ground-plane capacitance example.

This example exercises the image-series capacitance helper on a sweep that is
large enough to catch truncation mistakes:

* near the plane, the image series converges slowly and C rises well above the
  isolated-sphere value;
* far from the plane, C approaches 4*pi*eps0*a with the first correction a/(2h);
* the electrostatic attraction inferred from -dC/dh is positive.

Run:

    python examples/electrostatics/validation_sphere_ground_plane_capacitance.py
"""

from __future__ import annotations

import argparse
import json
import math
import sys
from pathlib import Path


HERE = Path(os.path.join(os.getcwd(), 'notebook.py')).resolve().parent
REPO = HERE.parents[1]
SRC = REPO / "packages" / "radia-mcp" / "src"
# radia_mcp is pip-installed (editable); no sys.path shim needed

from radia_mcp.radia_ngsolve.electrostatics import (  # noqa: E402
    EPS0,
    sphere_above_plane_capacitance,
)


OUT_JSON = HERE / "validation_sphere_ground_plane_capacitance_summary.json"
RADIUS_A = 0.01
EPS_R = 1.0
VOLTAGE = 100.0
HEIGHT_RATIOS = (1.05, 1.10, 1.25, 1.50, 2.0, 3.0, 5.0, 10.0, 25.0, 50.0, 100.0)
TERM_COUNTS = (5, 10, 20, 40, 80)
REFERENCE_TERMS = 1500


def _image_term(alpha: float, n: int) -> float:
    x = n * alpha
    numerator = math.sinh(alpha)
    if x < 40.0:
        return numerator / math.sinh(x)
    return 2.0 * numerator * math.exp(-x)


def reference_capacitance(radius_a: float, height_h: float, eps_r: float = 1.0,
                          n_terms: int = REFERENCE_TERMS) -> float:
    """Overflow-safe high-term image-series reference."""
    if not height_h > radius_a > 0.0:
        raise ValueError("require height_h > radius_a > 0")
    alpha = math.acosh(height_h / radius_a)
    series = sum(_image_term(alpha, n) for n in range(1, n_terms + 1))
    return 4.0 * math.pi * EPS0 * eps_r * radius_a * series


def _relative_error(value: float, reference: float) -> float:
    return abs(value - reference) / abs(reference)


def _finite_difference_force(radius_a: float, height_h: float, eps_r: float, voltage: float) -> dict:
    # At fixed voltage, W = 1/2 C V^2.  The attractive force magnitude is
    # -dW/dh = -1/2 V^2 dC/dh, positive because C decreases as h grows.
    step = max(1.0e-5 * height_h, 1.0e-6 * radius_a)
    c_minus = reference_capacitance(radius_a, height_h - step, eps_r)
    c_plus = reference_capacitance(radius_a, height_h + step, eps_r)
    dcdh = (c_plus - c_minus) / (2.0 * step)
    return {
        "height_m": height_h,
        "step_m": step,
        "dCdh_F_per_m": dcdh,
        "attractive_force_N": -0.5 * voltage * voltage * dcdh,
    }


def build_rows() -> list[dict]:
    c_isolated = 4.0 * math.pi * EPS0 * EPS_R * RADIUS_A
    rows = []
    for ratio in HEIGHT_RATIOS:
        height = ratio * RADIUS_A
        cref = reference_capacitance(RADIUS_A, height, EPS_R)
        by_terms = {}
        rel_errors = {}
        for n_terms in TERM_COUNTS:
            value = sphere_above_plane_capacitance(
                RADIUS_A,
                height,
                eps_r=EPS_R,
                n_terms=n_terms,
            )["C"]
            by_terms[str(n_terms)] = value
            rel_errors[str(n_terms)] = _relative_error(value, cref)
        rows.append({
            "height_over_radius": ratio,
            "height_m": height,
            "reference_C_F": cref,
            "C_over_isolated": cref / c_isolated,
            "excess_over_isolated": cref / c_isolated - 1.0,
            "capacitance_by_terms_F": by_terms,
            "relative_errors_by_terms": rel_errors,
        })
    return rows


def _far_field_checks(rows: list[dict]) -> list[dict]:
    checks = []
    for row in rows:
        ratio = row["height_over_radius"]
        if ratio < 25.0:
            continue
        asymptotic = 1.0 / (2.0 * ratio)
        actual = row["excess_over_isolated"]
        checks.append({
            "height_over_radius": ratio,
            "actual_excess": actual,
            "first_image_asymptotic": asymptotic,
            "relative_mismatch": abs(actual - asymptotic) / asymptotic,
        })
    return checks


def validate(rows: list[dict]) -> dict:
    normalized = [row["C_over_isolated"] for row in rows]
    force = _finite_difference_force(RADIUS_A, 2.0 * RADIUS_A, EPS_R, VOLTAGE)
    far_field = _far_field_checks(rows)
    checks = {
        "near_plane_C_over_isolated": normalized[0],
        "mid_sweep_C_over_isolated": next(row["C_over_isolated"] for row in rows
                                          if row["height_over_radius"] == 2.0),
        "far_C_over_isolated": normalized[-1],
        "monotone_decrease_with_height": all(a > b for a, b in zip(normalized, normalized[1:])),
        "max_rel_error_20_terms": max(row["relative_errors_by_terms"]["20"] for row in rows),
        "max_rel_error_40_terms": max(row["relative_errors_by_terms"]["40"] for row in rows),
        "max_rel_error_80_terms": max(row["relative_errors_by_terms"]["80"] for row in rows),
        "far_field": far_field,
        "force_at_height_over_radius_2": force,
    }
    assert checks["near_plane_C_over_isolated"] > 2.4
    assert checks["mid_sweep_C_over_isolated"] > 1.3
    assert abs(checks["far_C_over_isolated"] - 1.0) < 6.0e-3
    assert checks["monotone_decrease_with_height"]
    assert checks["max_rel_error_20_terms"] < 1.5e-3
    assert checks["max_rel_error_40_terms"] < 3.0e-6
    assert checks["max_rel_error_80_terms"] < 1.0e-10
    assert max(item["relative_mismatch"] for item in far_field) < 3.0e-2
    assert force["dCdh_F_per_m"] < 0.0
    assert force["attractive_force_N"] > 0.0
    return checks


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--out", type=Path, default=OUT_JSON)
    args = parser.parse_args()

    rows = build_rows()
    checks = validate(rows)
    summary = {
        "kind": "sphere_ground_plane_capacitance_validation",
        "validation_class": True,
        "radius_m": RADIUS_A,
        "eps_r": EPS_R,
        "reference_terms": REFERENCE_TERMS,
        "term_counts": list(TERM_COUNTS),
        "rows": rows,
        "checks": checks,
    }
    args.out.parent.mkdir(parents=True, exist_ok=True)
    args.out.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print("[sphere above grounded plane capacitance]")
    for row in rows:
        print(
            f"  h/a={row['height_over_radius']:6.2f}  "
            f"C/C_iso={row['C_over_isolated']:.9f}  "
            f"err20={row['relative_errors_by_terms']['20']:.3e}  "
            f"err40={row['relative_errors_by_terms']['40']:.3e}"
        )
    force = checks["force_at_height_over_radius_2"]
    print(
        "[checks] "
        f"near C/C_iso={checks['near_plane_C_over_isolated']:.6f}, "
        f"far C/C_iso={checks['far_C_over_isolated']:.6f}, "
        f"F(h/a=2,V={VOLTAGE:g})={force['attractive_force_N']:.6e} N"
    )
    print(f"[OK] wrote {args.out}")
    return 0


if True:
    main()


[sphere above grounded plane capacitance]
  h/a=  1.05  C/C_iso=2.467482913  err20=1.289e-03  err40=2.371e-06
  h/a=  1.10  C/C_iso=2.155086117  err20=1.069e-04  err40=1.501e-08
  h/a=  1.25  C/C_iso=1.778396200  err20=8.044e-07  err40=7.671e-13
  h/a=  1.50  C/C_iso=1.535370509  err20=3.933e-09  err40=0.000e+00
  h/a=  2.00  C/C_iso=1.341059813  err20=3.441e-12  err40=0.000e+00
  h/a=  3.00  C/C_iso=1.201155285  err20=6.044e-16  err40=0.000e+00
  h/a=  5.00  C/C_iso=1.111236084  err20=0.000e+00  err40=0.000e+00
  h/a= 10.00  C/C_iso=1.052638523  err20=0.000e+00  err40=0.000e+00
  h/a= 25.00  C/C_iso=1.020408330  err20=0.000e+00  err40=0.000e+00
  h/a= 50.00  C/C_iso=1.010101020  err20=0.000e+00  err40=0.000e+00
  h/a=100.00  C/C_iso=1.005025126  err20=0.000e+00  err40=0.000e+00
[checks] near C/C_iso=2.467483, far C/C_iso=1.005025, F(h/a=2,V=100)=1.342633e-07 N
[OK] wrote \\192.168.11.100\work\00_CAE\Radia\01_GitHub\docs\electrostatics\validation_sphere_ground_plane_capacitance_summary

## 5. Maxwell stress / traction

Normal-field interface pressure `p = 1/2 eps0 E^2` -- the electrostatic mirror of the magnetic Maxwell-traction validation, easy to compare against FEM post-processing.

In [5]:
import sys, os
sys.argv = ["notebook"]

"""Validation-class electrostatic Maxwell stress / traction identities.

For a field normal to a conductor/dielectric interface, the electrostatic
Maxwell stress gives the familiar pressure

    p = 0.5 * eps0 * E^2.

This example mirrors the magnetic Maxwell-traction validation in a form that is
easy to compare with electrostatic FEM post-processing.

Run:

    python examples/electrostatics/validation_electrostatic_maxwell_traction.py
"""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path


HERE = Path(os.path.join(os.getcwd(), 'notebook.py')).resolve().parent
REPO = HERE.parents[1]
SRC = REPO / "packages" / "radia-mcp" / "src"
# radia_mcp is pip-installed (editable); no sys.path shim needed

from radia_mcp.radia_ngsolve.force import (  # noqa: E402
    EPS0,
    electrostatic_stress_tensor,
    electrostatic_traction_summary,
)


OUT_JSON = HERE / "validation_electrostatic_maxwell_traction_summary.json"

CASES = [
    {
        "name": "normal_1MV_per_m",
        "E": [0.0, 0.0, 1.0e6],
        "normal": [0.0, 0.0, 1.0],
        "area_m2": 0.25,
    },
    {
        "name": "tangential_1MV_per_m",
        "E": [1.0e6, 0.0, 0.0],
        "normal": [0.0, 0.0, 1.0],
        "area_m2": 0.25,
    },
    {
        "name": "oblique_3_4MV_per_m",
        "E": [3.0e6, 4.0e6, 0.0],
        "normal": [1.0, 0.0, 0.0],
        "area_m2": 1.0e-4,
    },
]


def _assert_close(value: float, expected: float, rel: float = 1.0e-12, abs_tol: float = 1.0e-12) -> None:
    if abs(value - expected) > max(abs_tol, rel * max(abs(value), abs(expected), 1.0)):
        raise AssertionError(f"{value!r} != {expected!r}")


def build_rows() -> list[dict]:
    rows = []
    for case in CASES:
        tensor = electrostatic_stress_tensor(case["E"])
        traction = electrostatic_traction_summary(
            case["E"],
            case["normal"],
            area_m2=case["area_m2"],
        )
        rows.append({
            "name": case["name"],
            "E": case["E"],
            "normal": traction["normal"],
            "area_m2": case["area_m2"],
            "stress_tensor_Pa": tensor,
            "traction": traction,
        })
    return rows


def validate(rows: list[dict]) -> dict:
    by_name = {row["name"]: row for row in rows}
    pressure_1mv = 0.5 * EPS0 * 1.0e12
    checks = {
        "pressure_1MV_per_m_Pa": pressure_1mv,
        "normal_traction_Pa": by_name["normal_1MV_per_m"]["traction"]["normal_traction_Pa"],
        "normal_force_N": by_name["normal_1MV_per_m"]["traction"]["force_N"],
        "tangential_normal_traction_Pa": (
            by_name["tangential_1MV_per_m"]["traction"]["normal_traction_Pa"]
        ),
        "oblique_normal_traction_Pa": by_name["oblique_3_4MV_per_m"]["traction"]["normal_traction_Pa"],
        "oblique_tangential_traction_magnitude_Pa": (
            by_name["oblique_3_4MV_per_m"]["traction"]["tangential_traction_magnitude_Pa"]
        ),
    }
    _assert_close(checks["normal_traction_Pa"], pressure_1mv)
    _assert_close(checks["normal_force_N"][2], 0.25 * pressure_1mv)
    _assert_close(checks["tangential_normal_traction_Pa"], -pressure_1mv)
    _assert_close(checks["oblique_normal_traction_Pa"], 0.5 * EPS0 * (9.0e12 - 16.0e12))
    _assert_close(checks["oblique_tangential_traction_magnitude_Pa"], EPS0 * 12.0e12)
    return checks


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--out", type=Path, default=OUT_JSON)
    args = parser.parse_args()

    rows = build_rows()
    checks = validate(rows)
    summary = {
        "kind": "electrostatic_maxwell_traction_identities",
        "validation_class": True,
        "eps0": EPS0,
        "rows": rows,
        "checks": checks,
    }
    args.out.parent.mkdir(parents=True, exist_ok=True)
    args.out.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")

    print("[Electrostatic Maxwell traction identities]")
    for row in rows:
        traction = row["traction"]
        print(
            f"  {row['name']}: normal={traction['normal_traction_Pa']:.6g} Pa, "
            f"tangent={traction['tangential_traction_magnitude_Pa']:.6g} Pa"
        )
    print("[checks]")
    for key, value in checks.items():
        print(f"  {key}: {value}")
    return 0


if True:
    main()


[Electrostatic Maxwell traction identities]
  normal_1MV_per_m: normal=4.42709 Pa, tangent=0 Pa
  tangential_1MV_per_m: normal=-4.42709 Pa, tangent=0 Pa
  oblique_3_4MV_per_m: normal=-30.9897 Pa, tangent=106.25 Pa
[checks]
  pressure_1MV_per_m_Pa: 4.4270939064000006
  normal_traction_Pa: 4.4270939064000006
  normal_force_N: [0.0, 0.0, 1.1067734766000001]
  tangential_normal_traction_Pa: -4.4270939064000006
  oblique_normal_traction_Pa: -30.9896573448
  oblique_tangential_traction_magnitude_Pa: 106.2502537536
